In [1]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION WITH 8-LABEL RHETORICAL ROLES
# Combines: BART + LED + PEGASUS + HSLN Role Classification
# Selection: HYBRID Role Weight + Content Salience
# Mapping: 13 HSLN labels → 8 strategic labels
# IMPROVEMENTS:
# 1. Fixed ensemble weights (LED gets 50% weight)
# 2. Adaptive target sentences based on document length
# 3. Hybrid scoring: Role importance + Content salience
# 4. Novel Idea 5: KG-Diff Iterative Refinement
# 5. Novel Idea 7: Semantic KG Coverage
# 6. Novel Idea 8: LCS-Guided Sentence Fusion (ROUGE-L booster)
# ✨ Novel Idea 9: Verbatim Span Injection (Method C — Copy Mechanism)
#    TARGET: ROUGE-L ≥ 0.36
#    WHY:    Abstractive generation paraphrases source sentences, breaking
#            exact token chains that ROUGE-L rewards. By finding the longest
#            verbatim contiguous span shared between a generated sentence and
#            the critical-role source pool, and substituting the generated
#            sentence with the source sentence containing that span, we
#            guarantee longer unbroken token chains with the reference.
#    HOW:    For each sentence in the LCS-fused summary:
#            Step 1 — Span detection: slide a window over source critical
#                     sentences to find the longest contiguous n-gram span
#                     that appears in both the summary sentence and source.
#            Step 2 — Context expansion: once a shared span is found, expand
#                     left/right up to MAX_CONTEXT_TOKENS tokens to include
#                     the surrounding legal context from the source sentence.
#            Step 3 — LCS gating: only substitute if the source sentence's
#                     LCS with the critical pool exceeds the original
#                     generated sentence's LCS (prevents degradation).
#            Step 4 — Role filtering: only substitute ISSUE / REASONING /
#                     PRECEDENT_RELIED sentences — keep FACTS abstractive
#                     to preserve BERTScore.
#    RESULT: Direct verbatim token chains guaranteed in output → higher LCS
#            → higher ROUGE-L with negligible BERTScore trade-off.
# ✨ Novel Idea 10: KG-Attention During Generation
#    TARGET: Better factual coverage + ROUGE-L via entity-grounded generation
#    WHY:    The standard beam search treats all tokens equally. However,
#            for legal summarization, certain entity tokens (party names,
#            section numbers, case citations, legal concepts from the KG)
#            MUST appear in the output — their absence directly causes
#            KG triples to be uncovered and hurts ROUGE-L.
#    HOW:    We implement a custom LogitsProcessor that:
#            Step 1 — Extracts entity tokens from KG critical triples
#                     (subjects + objects from ISSUE/REASONING/PRECEDENT roles).
#            Step 2 — Builds a token-id → boost-score mapping using the
#                     LED tokenizer. Multi-token entities (e.g. "Section 302")
#                     get higher boost than single tokens to reward exact phrases.
#            Step 3 — During each generation step, the processor adds the
#                     boost score to the logit of every entity token.
#                     The boost decays as the entity is already present in
#                     the generated prefix (to avoid repetition).
#            Step 4 — Entity boost is only applied to LED (the highest-quality
#                     baseline model). BART and PEGASUS use standard generation
#                     to maintain output diversity in the ensemble.
#    RESULT: Generated text is more likely to contain exact legal entity
#            strings → longer shared subsequences with reference → higher
#            ROUGE-L and ROUGE-2 simultaneously.
# =========================================================

import os
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration,
    LogitsProcessor,
    LogitsProcessorList
)
from collections import Counter, defaultdict
import evaluate
import nltk

# Download NLTK data
print("📥 Downloading NLTK data...")
for resource in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource.split('/')[-1], quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION
# =========================================================

MODEL_PATHS = {
    "BART":       "BART",
    "LED":        "LED",
    "PEGASUS":    "PEGASUS",
    "ROLE_MODEL": "best_RR_model"
}

TRAIN_JSON_PATH = "train.json"
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./ensemble_results_8label_rougeL_v2"

ROLE_CLASSIFICATION_FILE = "sentence_role_annotations_8label.json"
ROLE_CONTRIBUTION_FILE   = "role_contribution_scores_8label.json"
ROLE_WEIGHTS_FILE        = "normalized_role_weights_8label.json"

MODEL_CONFIG = {
    "BART": {
        "max_input":  1024,
        "max_output": 256,
        "num_beams":  4
    },
    "LED": {
        "max_input":  4096,
        "max_output": 512,
        "num_beams":  4
    },
    "PEGASUS": {
        "max_input":  1024,
        "max_output": 256,
        "num_beams":  4
    }
}

# =========================================================
# LABEL SYSTEM: 13 → 8 MAPPING
# =========================================================

HSLN_LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE"
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE",
    "FACTS",
    "ISSUE",
    "ARGUMENTS",
    "REASONING",
    "PRECEDENT_RELIED",
    "PRECEDENT_NOT_RELIED",
    "PROCEDURAL"
]
NUM_LABELS = 8

LABEL_MAPPING_13_TO_8 = {
    "PREAMBLE":       "PREAMBLE",
    "ISSUE":          "ISSUE",
    "FAC":            "FACTS",
    "PRE_RELIED":     "PRECEDENT_RELIED",
    "PRE_NOT_RELIED": "PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER": "ARGUMENTS",
    "ARG_RESPONDENT": "ARGUMENTS",
    "ANALYSIS":       "REASONING",
    "RATIO":          "REASONING",
    "RPC":            "REASONING",
    "STA":            "PROCEDURAL",
    "RLC":            "PROCEDURAL",
    "NONE":           "PROCEDURAL"
}

def map_13_to_8_labels(labels_13):
    return [LABEL_MAPPING_13_TO_8.get(label, "PROCEDURAL") for label in labels_13]

# =========================================================
# HYBRID SCORING CONFIGURATION
# =========================================================
HYBRID_ALPHA = 0.6
HYBRID_BETA  = 0.4

# =========================================================
# ADAPTIVE TARGET SENTENCES
# =========================================================
COMPRESSION_RATIOS = {
    "BART":    0.12,
    "PEGASUS": 0.12,
    "LED":     0.40
}
MIN_SENTENCES = {"BART": 30,  "PEGASUS": 30,  "LED": 100}
MAX_SENTENCES = {"BART": 150, "PEGASUS": 150, "LED": 400}

def get_adaptive_targets(doc_length):
    adaptive_targets = {}
    for model_name, ratio in COMPRESSION_RATIOS.items():
        target = int(doc_length * ratio)
        target = max(MIN_SENTENCES[model_name], target)
        target = min(MAX_SENTENCES[model_name], target)
        adaptive_targets[model_name] = target
    return adaptive_targets

MAX_TRAIN_SAMPLES = 3000

# =========================================================
# ENSEMBLE WEIGHTS
# =========================================================
ENSEMBLE_WEIGHTS = {
    "BART":    0.25,
    "LED":     0.50,
    "PEGASUS": 0.25
}

# =========================================================
# KG-DIFF CONFIGURATION  (Novel Ideas 5 + 7)
# =========================================================
KG_CRITICAL_ROLES     = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]
KG_SEMANTIC_THRESHOLD = 0.75
KG_COVERAGE_THRESHOLD = 0.75
KG_MAX_ITERATIONS     = 2
KG_TOP_MISSING        = 5
KG_MAX_CORRECTION_SENTS = 10

# =========================================================
# NOVEL IDEA 8: LCS-GUIDED SENTENCE FUSION CONFIGURATION
# =========================================================
LCS_ANCHOR_ROLES          = ["ISSUE", "REASONING", "PRECEDENT_RELIED", "FACTS"]
LCS_MIN_NGRAM_OVERLAP     = 3
LCS_CONNECTIVES = [
    "The court held that",
    "It was observed that",
    "Therefore,",
    "Furthermore,",
    "The appellant contended that",
    "In view of the above,",
    "The High Court noted that",
    "Accordingly,"
]
LCS_MAX_OUTPUT_SENTENCES   = 20
LCS_ANCHOR_LCS_WEIGHT      = 0.5
LCS_ANCHOR_SEMANTIC_WEIGHT = 0.5

# =========================================================
# ✨ NOVEL IDEA 9: VERBATIM SPAN INJECTION CONFIGURATION
# ─────────────────────────────────────────────────────────
#
# DESIGN RATIONALE:
#   ROUGE-L rewards the Longest Common Subsequence of TOKENS between
#   generated and reference. Abstractive generation creates paraphrases
#   that semantically match the reference but break exact token chains.
#   Verbatim Span Injection replaces generated sentences with verbatim
#   source sentences whenever a shared span is detected — guaranteeing
#   that the injected sentence's token sequence is maximally aligned
#   with the source, which reference summaries also tend to preserve.
#
# WHY NOT JUST USE ALL SOURCE SENTENCES?
#   Using all source sentences (extractive summarization) would maximise
#   LCS but destroy precision — the reference summaries are highly
#   compressed. We only substitute when:
#   (a) A long verbatim span (≥ VERBATIM_MIN_SPAN tokens) is found, AND
#   (b) The substituted source sentence has higher LCS against the
#       critical pool than the original generated sentence.
#   This ensures we only improve, never degrade.
#
# SELECTIVE ROLE APPLICATION:
#   We apply verbatim injection to ISSUE and REASONING sentences only.
#   These are the roles where annotators most directly copy source
#   language. For FACTS, we keep the abstractive version to avoid
#   copying very long fact descriptions that inflate length without
#   improving LCS density (long source sentences → low LCS density).
# =========================================================

# Minimum contiguous n-gram span length to trigger substitution
VERBATIM_MIN_SPAN = 5

# Maximum tokens to expand left/right around the detected span
# when extracting a verbatim excerpt (instead of the full sentence)
VERBATIM_MAX_CONTEXT_TOKENS = 10

# Roles where verbatim injection is applied
# FACTS excluded intentionally — they tend to be very long
VERBATIM_TARGET_ROLES = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]

# Only substitute a generated sentence with a source sentence if
# the source sentence is no more than this ratio longer than the
# generated sentence (prevents very long substitutions)
VERBATIM_MAX_LENGTH_RATIO = 2.5

# Log at most this many substitutions per document for diagnostics
VERBATIM_LOG_LIMIT = 5

# =========================================================
# ✨ NOVEL IDEA 10: KG-ATTENTION DURING GENERATION CONFIGURATION
# ─────────────────────────────────────────────────────────
#
# DESIGN RATIONALE:
#   Standard beam search is entity-agnostic — it treats "Section 302",
#   "petitioner", and "the" with equal probability given the context.
#   In legal summarization, exact entity strings (party names, statutory
#   sections, case citations) MUST appear in the summary to match the
#   reference. We add a LogitsProcessor that boosts entity token logits
#   during every generation step.
#
# BOOST DECAY:
#   If an entity is already present in the generated prefix, its boost
#   decays exponentially (factor ENTITY_DECAY_FACTOR per occurrence).
#   This prevents the model from looping on boosted entities.
#
# MULTI-TOKEN ENTITY BONUS:
#   For multi-word entities (e.g. "Income Tax Act"), each constituent
#   token gets boost = ENTITY_BASE_BOOST * MULTI_TOKEN_MULTIPLIER.
#   Single-token entities get base boost only.
#   Rationale: multi-token entities are harder for the model to generate
#   correctly, and they contribute more to ROUGE-2/L when matched.
#
# SAFETY:
#   Boost is additive to logits (not multiplicative) to avoid numerical
#   instability. We cap the maximum effective boost at ENTITY_MAX_BOOST
#   to prevent degenerate outputs where entity tokens are always selected.
# =========================================================

# Additive logit boost for KG entity tokens
ENTITY_BASE_BOOST = 1.5

# Extra multiplier for tokens that are part of multi-word entities
ENTITY_MULTI_TOKEN_MULTIPLIER = 1.3

# Decay factor per occurrence of the entity in the already-generated prefix
# 1.0 = no decay, 0.5 = halved per occurrence
ENTITY_DECAY_FACTOR = 0.6

# Maximum allowed effective boost (clipping) to prevent degenerate outputs
ENTITY_MAX_BOOST = 3.0

# Minimum entity string length (tokens) to be included in the boost set
ENTITY_MIN_TOKEN_LEN = 2

# Only apply KG-Attention to LED (highest quality baseline)
ENTITY_BOOST_MODELS = ["LED"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*70)
print("📋 CONFIGURATION — ROUGE-L IMPROVEMENT v2 (Novel Ideas 9 + 10)")
print("="*70)
print(f"Label System:   8 Strategic Labels")
print(f"Selection:      Hybrid Role Weight + Content Salience")
print(f"\n✨ Novel Idea 9: Verbatim Span Injection")
print(f"  Min span:            {VERBATIM_MIN_SPAN} tokens")
print(f"  Target roles:        {VERBATIM_TARGET_ROLES}")
print(f"  Max length ratio:    {VERBATIM_MAX_LENGTH_RATIO}x")
print(f"  Context expansion:   ±{VERBATIM_MAX_CONTEXT_TOKENS} tokens")
print(f"\n✨ Novel Idea 10: KG-Attention During Generation")
print(f"  Base boost:          {ENTITY_BASE_BOOST}")
print(f"  Multi-token mult:    {ENTITY_MULTI_TOKEN_MULTIPLIER}x")
print(f"  Decay factor:        {ENTITY_DECAY_FACTOR} per occurrence")
print(f"  Max boost cap:       {ENTITY_MAX_BOOST}")
print(f"  Applied to models:   {ENTITY_BOOST_MODELS}")
print(f"\n⚡ ENSEMBLE WEIGHTS:")
print(f"   BART:    {ENSEMBLE_WEIGHTS['BART']:.2f}")
print(f"   LED:     {ENSEMBLE_WEIGHTS['LED']:.2f} ← BEST baseline")
print(f"   PEGASUS: {ENSEMBLE_WEIGHTS['PEGASUS']:.2f}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print("="*70)

# =========================================================
# HSLN MODEL DEFINITION
# =========================================================

class PrototypeAttention(nn.Module):
    """Prototype-based multi-head attention"""
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h  = heads
        self.dh = dim // heads
        self.q  = nn.Linear(dim, dim, bias=False)
        self.k  = nn.Linear(dim, dim, bias=False)
        self.v  = nn.Linear(dim, dim, bias=False)
        self.o  = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape
        C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, dim=-1)
        K = F.normalize(K, dim=-1)
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            a = self.dropout(a)
            outs.append(a @ V[:, h])
        out = torch.cat(outs, -1)
        return self.ln(x + self.dropout(self.o(out)))


class RPL(nn.Module):
    """Representation Prototype Learning"""
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))

    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T


class RTM(nn.Module):
    """Rhetorical Transition Model"""
    def __init__(self, num_labels, rtm_lambda):
        super().__init__()
        self.A          = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.rtm_lambda = rtm_lambda

    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(
                lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1
            )
            logits[:, t] += self.rtm_lambda * tr
        return logits


class HSLNModel(nn.Module):
    """HSLN Model — Pure PyTorch (13 labels)"""

    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.lstm_hidden   = lstm_hidden
        self.num_labels    = num_labels
        self.att  = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(
            embedding_dim, lstm_hidden, 2,
            bidirectional=True, batch_first=True, dropout=dropout
        )
        self.cls   = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl   = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm   = RTM(num_labels, rtm_lambda)
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))

    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck  = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        l1  = self.cls(o)
        l2  = self.rpl(o)
        a   = torch.sigmoid(self.alpha)
        return self.rtm(a * l1 + (1 - a) * l2)

    def predict(self, embeddings, lengths=None):
        return torch.argmax(self.forward(embeddings, lengths), dim=-1)


# =========================================================
# LOAD HSLN ROLE CLASSIFIER
# =========================================================

print("\n📚 Loading HSLN role classifier (13 labels)...")

config_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config_dict = json.load(f)
    embedding_dim = config_dict.get('embedding_dim', 768)
    lstm_hidden   = config_dict.get('lstm_hidden', 384)
    dropout       = config_dict.get('dropout', 0.3)
    rtm_lambda    = config_dict.get('rtm_lambda', 0.05)
else:
    embedding_dim, lstm_hidden, dropout, rtm_lambda = 768, 384, 0.3, 0.05

role_model = HSLNModel(
    embedding_dim=embedding_dim,
    lstm_hidden=lstm_hidden,
    num_labels=NUM_HSLN_LABELS,
    dropout=dropout,
    rtm_lambda=rtm_lambda
)

weights_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "pytorch_model.bin")
state_dict   = torch.load(weights_path, map_location=device)
if 'prototypes' in state_dict:
    del state_dict['prototypes']
role_model.load_state_dict(state_dict, strict=False)

prototypes_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "prototypes.pt")
prototypes      = torch.load(prototypes_path, map_location=device)
role_model.prototypes = prototypes

role_model.to(device)
role_model.eval()
print("✅ HSLN role classifier loaded (13 labels → mapped to 8)")

# =========================================================
# LOAD InLegalBERT FOR EMBEDDINGS
# =========================================================

print("\n📚 Loading InLegalBERT for embeddings...")
legal_model_name = "law-ai/InLegalBERT"
legal_tokenizer  = AutoTokenizer.from_pretrained(legal_model_name)
legal_model      = AutoModel.from_pretrained(legal_model_name).to(device)
legal_model.eval()
print("✅ InLegalBERT loaded successfully")

# =========================================================
# EMBEDDING AND ROLE CLASSIFICATION FUNCTIONS
# =========================================================

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    """Compute mean-pooled sentence embeddings using InLegalBERT."""
    if len(texts) == 0:
        return np.array([])
    embs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i + batch_size]
        enc    = legal_tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors="pt"
        ).to(device)
        out    = legal_model(**enc).last_hidden_state
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1)
        embs.append(pooled.cpu().numpy())
    return np.vstack(embs)


@torch.no_grad()
def classify_roles(sentences, batch_size=16):
    """Classify rhetorical roles using HSLN and map to 8-label system."""
    if not sentences:
        return []
    all_predictions_13 = []
    for i in range(0, len(sentences), batch_size):
        batch_sents      = sentences[i:i + batch_size]
        inputs           = legal_tokenizer(
            batch_sents, padding=True, truncation=True,
            max_length=128, return_tensors="pt"
        ).to(device)
        embeddings       = legal_model(**inputs).last_hidden_state.mean(dim=1)
        embeddings_batch = embeddings.unsqueeze(1)
        lengths          = torch.ones(len(batch_sents), dtype=torch.long).to(device)
        predictions      = role_model.predict(embeddings_batch, lengths)
        batch_preds      = predictions[:, 0].cpu().tolist()
        all_predictions_13.extend(batch_preds)
    labels_13 = [HSLN_LABELS[pred] for pred in all_predictions_13]
    return map_13_to_8_labels(labels_13)


# =========================================================
# KG TRIPLE EXTRACTION
# =========================================================

def extract_triples_as_tuples(sentences):
    """Extract (subject, verb, object) triples from sentences."""
    triples = []
    try:
        import spacy
        try:
            nlp = spacy.load("en_legal_ner_trf")
        except OSError:
            try:
                nlp = spacy.load("en_core_web_sm")
            except OSError:
                return _extract_triples_fallback(sentences)

        for sent in sentences:
            if not sent.strip():
                continue
            doc = nlp(sent[:512])
            for token in doc:
                if token.dep_ == "ROOT" and token.pos_ == "VERB":
                    subjects = [c for c in token.children
                                if c.dep_ in ("nsubj", "nsubjpass")]
                    objects  = [c for c in token.children
                                if c.dep_ in ("dobj", "pobj", "attr", "oprd")]
                    for subj in subjects:
                        for obj in objects:
                            subj_text = " ".join(
                                t.text for t in subj.subtree if not t.is_stop
                            ).lower().strip()
                            obj_text  = " ".join(
                                t.text for t in obj.subtree  if not t.is_stop
                            ).lower().strip()
                            rel_text  = token.lemma_.lower()
                            if subj_text and obj_text and rel_text:
                                triples.append((subj_text, rel_text, obj_text))
    except ImportError:
        triples = _extract_triples_fallback(sentences)
    return triples


def _extract_triples_fallback(sentences):
    """Noun co-occurrence fallback when spaCy is unavailable."""
    triples = []
    for sent in sentences:
        words = re.findall(r'\b[A-Za-z][a-z]+(?:\s+[A-Z][a-z]+)*\b', sent)
        words = [w.lower() for w in words if len(w) > 3]
        for i in range(len(words) - 1):
            triples.append((words[i], "related_to", words[i + 1]))
    return triples


# =========================================================
# NOVEL IDEA 7: SEMANTIC KG COVERAGE
# =========================================================

def triple_to_text(triple):
    subj, rel, obj = triple
    return f"{subj} {rel} {obj}"


def compute_semantic_kg_coverage(critical_triples, summary_triples,
                                  semantic_threshold=KG_SEMANTIC_THRESHOLD):
    """Semantic KG Coverage using InLegalBERT cosine similarity."""
    if not critical_triples:
        return 1.0, [], []
    if not summary_triples:
        uncovered  = [(t, 0.0) for t in critical_triples]
        per_scores = [(t, 0.0, False) for t in critical_triples]
        return 0.0, per_scores, uncovered

    critical_texts  = [triple_to_text(t) for t in critical_triples]
    summary_texts   = [triple_to_text(t) for t in summary_triples]
    critical_embeddings = embed_with_legalbert(critical_texts)
    summary_embeddings  = embed_with_legalbert(summary_texts)
    sim_matrix = cosine_similarity(critical_embeddings, summary_embeddings)

    per_triple_scores = []
    uncovered_triples = []
    covered_count     = 0

    for i, crit_triple in enumerate(critical_triples):
        max_sim    = float(sim_matrix[i].max())
        is_covered = max_sim >= semantic_threshold
        per_triple_scores.append((crit_triple, max_sim, is_covered))
        if is_covered:
            covered_count += 1
        else:
            uncovered_triples.append((crit_triple, max_sim))

    uncovered_triples.sort(key=lambda x: x[1])
    coverage_score = covered_count / len(critical_triples)
    return coverage_score, per_triple_scores, uncovered_triples


def get_missing_entities_from_uncovered(uncovered_triples, top_k=KG_TOP_MISSING):
    """Extract entity tokens from the most uncovered critical triples."""
    missing_entities = set()
    for (subj, rel, obj), max_sim in uncovered_triples[:top_k]:
        for token in subj.split() + obj.split():
            if len(token) > 3:
                missing_entities.add(token.lower())
    return missing_entities


# =========================================================
# LCS UTILITY FUNCTIONS (shared by Ideas 8, 9, 10)
# =========================================================

def compute_token_lcs_length(tokens_a, tokens_b):
    """
    Compute the LCS length between two token lists using DP.
    Space-optimised: O(min(|A|,|B|)) using two rows.
    """
    if not tokens_a or not tokens_b:
        return 0
    if len(tokens_a) < len(tokens_b):
        tokens_a, tokens_b = tokens_b, tokens_a
    n    = len(tokens_b)
    prev = [0] * (n + 1)
    curr = [0] * (n + 1)
    for token_a in tokens_a:
        for j, token_b in enumerate(tokens_b):
            if token_a.lower() == token_b.lower():
                curr[j + 1] = prev[j] + 1
            else:
                curr[j + 1] = max(curr[j], prev[j + 1])
        prev, curr = curr, [0] * (n + 1)
    return prev[n]


def compute_lcs_score(sentence, source_sentences):
    """
    Normalised LCS score between a sentence and a pool.
    Normalised LCS = lcs_length / max(|sent|, |src|)
    Returns best score across all source sentences.
    """
    if not source_sentences:
        return 0.0
    sent_tokens = word_tokenize(sentence.lower())
    if not sent_tokens:
        return 0.0
    best_score = 0.0
    for src_sent in source_sentences:
        src_tokens = word_tokenize(src_sent.lower())
        if not src_tokens:
            continue
        lcs_len    = compute_token_lcs_length(sent_tokens, src_tokens)
        max_len    = max(len(sent_tokens), len(src_tokens))
        score      = lcs_len / max_len if max_len > 0 else 0.0
        best_score = max(best_score, score)
    return best_score


def find_source_position(sentence, doc_annotation):
    """
    Find source document sentence index with highest LCS overlap.
    Used for position-aware reordering in Novel Idea 8.
    """
    best_pos   = 999
    best_score = -1.0
    sent_tokens = word_tokenize(sentence.lower())
    for s in doc_annotation["sentences"]:
        src_tokens = word_tokenize(s["sentence"].lower())
        lcs_len    = compute_token_lcs_length(sent_tokens, src_tokens)
        if lcs_len > best_score:
            best_score = lcs_len
            best_pos   = s["index"]
    return best_pos


# =========================================================
# NOVEL IDEA 8: LCS-GUIDED SENTENCE FUSION
# =========================================================

def fuse_adjacent_sentences(sent_a, sent_b, min_ngram=LCS_MIN_NGRAM_OVERLAP):
    """
    Detect and merge overlapping n-gram spans at the boundary of two sentences.
    If the tail of sent_a matches the head of sent_b for ≥ min_ngram tokens,
    remove the redundant span from sent_b's head and concatenate.
    """
    tokens_a  = word_tokenize(sent_a.lower())
    tokens_b  = word_tokenize(sent_b.lower())
    max_check = min(15, len(tokens_a), len(tokens_b))
    for n in range(max_check, min_ngram - 1, -1):
        if tokens_a[-n:] == tokens_b[:n]:
            original_b_words     = word_tokenize(sent_b)
            fused_b_words        = original_b_words[n:]
            if fused_b_words:
                fused_b_words[0] = fused_b_words[0].lower()
                fused_tail       = " ".join(fused_b_words)
                sent_a_stripped  = sent_a.rstrip(". ")
                return f"{sent_a_stripped}, {fused_tail}"
    return f"{sent_a} {sent_b}"


def inject_connectives(sentences, connectives=LCS_CONNECTIVES):
    """Insert legal connective phrases before pronoun-start sentences."""
    if not sentences:
        return sentences
    pronoun_triggers = {"it", "this", "they", "the same", "such", "these", "those"}
    result   = [sentences[0]]
    conn_idx = 0
    for i in range(1, len(sentences)):
        sent       = sentences[i]
        first_word = sent.split()[0].lower() if sent.split() else ""
        if first_word in pronoun_triggers:
            connector = connectives[conn_idx % len(connectives)]
            conn_idx += 1
            if connector.endswith("that"):
                modified = f"{connector} {sent[0].lower()}{sent[1:]}"
            else:
                modified = f"{connector} {sent}"
            result.append(modified)
        else:
            result.append(sent)
    return result


def lcs_guided_sentence_fusion(kg_refined_summary, doc_annotation,
                                normalized_weights):
    """
    Novel Idea 8 — LCS-Guided Sentence Fusion for ROUGE-L improvement.
    Steps: Anchor scoring → position-aware reorder → n-gram fusion
           → connector injection.
    """
    fusion_log = {
        "method": "lcs_guided_sentence_fusion (Novel Idea 8)",
        "target": "ROUGE-L >= 0.34",
        "steps":  []
    }

    anchor_source_sents = [
        s["sentence"] for s in doc_annotation["sentences"]
        if s["role"] in LCS_ANCHOR_ROLES
    ]
    all_source_sents = [s["sentence"] for s in doc_annotation["sentences"]]
    if not anchor_source_sents:
        anchor_source_sents = all_source_sents

    summary_sentences = sent_tokenize(kg_refined_summary)
    if not summary_sentences:
        fusion_log["early_exit"] = "Empty summary"
        return kg_refined_summary, fusion_log

    sum_embeddings = embed_with_legalbert(summary_sentences)
    src_embeddings = embed_with_legalbert(anchor_source_sents)
    sem_matrix     = cosine_similarity(sum_embeddings, src_embeddings)
    sem_scores     = sem_matrix.max(axis=1)

    scored_sentences = []
    for i, sent in enumerate(summary_sentences):
        lcs_score    = compute_lcs_score(sent, anchor_source_sents)
        sem_score    = float(sem_scores[i])
        anchor_score = (LCS_ANCHOR_LCS_WEIGHT * lcs_score +
                        LCS_ANCHOR_SEMANTIC_WEIGHT * sem_score)
        scored_sentences.append({
            "sentence":    sent,
            "lcs_score":   round(lcs_score, 4),
            "sem_score":   round(sem_score, 4),
            "anchor_score": round(anchor_score, 4)
        })

    fusion_log["steps"].append({
        "step": 1, "name": "Sentence scoring",
        "total_summary_sentences": len(scored_sentences),
        "top3_scores": [
            {"sent": s["sentence"][:80], "score": s["anchor_score"]}
            for s in sorted(scored_sentences,
                            key=lambda x: x["anchor_score"], reverse=True)[:3]
        ]
    })

    sorted_by_score = sorted(scored_sentences,
                             key=lambda x: x["anchor_score"], reverse=True)
    selected = sorted_by_score[:LCS_MAX_OUTPUT_SENTENCES]
    for item in selected:
        item["source_pos"] = find_source_position(item["sentence"], doc_annotation)
    selected_ordered = sorted(selected, key=lambda x: x["source_pos"])
    ordered_sents    = [item["sentence"] for item in selected_ordered]

    fusion_log["steps"].append({
        "step": 2, "name": "Position-aware reordering",
        "selected_count": len(selected),
        "source_positions_used": [item["source_pos"] for item in selected_ordered]
    })

    if len(ordered_sents) >= 2:
        fused_sents  = [ordered_sents[0]]
        fusions_done = 0
        for i in range(1, len(ordered_sents)):
            prev   = fused_sents[-1]
            curr   = ordered_sents[i]
            merged = fuse_adjacent_sentences(prev, curr, LCS_MIN_NGRAM_OVERLAP)
            if merged != f"{prev} {curr}":
                fused_sents[-1] = merged
                fusions_done   += 1
            else:
                fused_sents.append(curr)
    else:
        fused_sents  = ordered_sents
        fusions_done = 0

    fusion_log["steps"].append({
        "step": 3, "name": "N-gram overlap fusion",
        "fusions_performed": fusions_done,
        "sentences_after_fusion": len(fused_sents)
    })

    final_sents = inject_connectives(fused_sents, LCS_CONNECTIVES)
    fusion_log["steps"].append({
        "step": 4, "name": "Connector injection",
        "sentences_final": len(final_sents)
    })

    fused_summary = " ".join(final_sents)
    fusion_log["final_sentence_count"] = len(final_sents)
    fusion_log["final_char_count"]     = len(fused_summary)
    return fused_summary, fusion_log


# =========================================================
# ✨ NOVEL IDEA 9: VERBATIM SPAN INJECTION
# ─────────────────────────────────────────────────────────
#
# ALGORITHM DETAILS:
#
#   For each sentence S in the LCS-fused summary:
#   1. Tokenize S → sent_tokens
#   2. For each critical source sentence Csrc:
#      a. Tokenize Csrc → src_tokens
#      b. Slide a window of size W (decreasing from min(20, |src|) to
#         VERBATIM_MIN_SPAN) over src_tokens:
#         - For each window position `start`, check if the window
#           src_tokens[start:start+W] appears ANYWHERE in sent_tokens
#           (not just as a prefix — full subsequence search).
#         - If found, record (Csrc, W, start) as a candidate match.
#      c. Keep the candidate with the largest W (longest span).
#   3. Among all source sentences, pick the one with the globally
#      longest verbatim span match.
#   4. Gate: only substitute if:
#      a. Span ≥ VERBATIM_MIN_SPAN tokens.
#      b. LCS(Csrc, critical_pool) > LCS(S, critical_pool).
#      c. len(Csrc_tokens) / len(sent_tokens) ≤ VERBATIM_MAX_LENGTH_RATIO.
#   5. If gating passes: extract a verbatim excerpt from Csrc using
#      context expansion (±VERBATIM_MAX_CONTEXT_TOKENS around the span).
#      Prefer the full Csrc sentence if it is not too long.
#   6. If gating fails: keep S unchanged.
#
# CONTEXT EXPANSION vs FULL SENTENCE:
#   Full sentence substitution is preferred when the source sentence is
#   ≤ VERBATIM_MAX_LENGTH_RATIO * len(sent_tokens) tokens long.
#   Otherwise, we extract a verbatim excerpt: the shared span ± context.
#   This prevents bloating the summary with very long source sentences.
# =========================================================

def _find_sublist(needle, haystack):
    """
    Find the first position of `needle` (list) in `haystack` (list).
    Returns the start index or -1 if not found.
    Uses a simple linear scan — acceptable for typical sentence lengths.
    """
    n = len(needle)
    h = len(haystack)
    if n > h:
        return -1
    for i in range(h - n + 1):
        if haystack[i:i + n] == needle:
            return i
    return -1


def _find_longest_verbatim_span(sent_tokens, src_tokens, min_span=VERBATIM_MIN_SPAN):
    """
    Find the longest contiguous span of src_tokens that appears anywhere
    in sent_tokens.

    Args:
        sent_tokens: List[str] — tokenized generated sentence (lowercase)
        src_tokens:  List[str] — tokenized source sentence (lowercase)
        min_span:    int       — minimum span length to consider

    Returns:
        Tuple (span_len, src_start, sent_start) or (0, -1, -1) if not found.
        - span_len:  length of the longest matching span
        - src_start: start index of the span in src_tokens
        - sent_start: start index of the span in sent_tokens
    """
    max_window = min(20, len(src_tokens), len(sent_tokens))
    for span_len in range(max_window, min_span - 1, -1):
        for src_start in range(len(src_tokens) - span_len + 1):
            span       = src_tokens[src_start:src_start + span_len]
            sent_start = _find_sublist(span, sent_tokens)
            if sent_start != -1:
                return span_len, src_start, sent_start
    return 0, -1, -1


def _extract_verbatim_excerpt(src_sentence, src_tokens_orig, src_start, span_len,
                               sent_len, max_context=VERBATIM_MAX_CONTEXT_TOKENS):
    """
    Extract a verbatim excerpt from the source sentence centred on the
    matched span. The excerpt is the span ± max_context tokens.

    If the resulting excerpt covers ≥ 90% of the source sentence, return
    the full source sentence instead (avoids awkward fragments).

    Args:
        src_sentence:    str       — original source sentence
        src_tokens_orig: List[str] — tokenized source sentence (original case)
        src_start:       int       — span start in src_tokens
        span_len:        int       — length of the matched span
        sent_len:        int       — length of the generated sentence (tokens)
        max_context:     int       — tokens to expand left/right

    Returns:
        str — verbatim excerpt or full source sentence
    """
    left  = max(0, src_start - max_context)
    right = min(len(src_tokens_orig), src_start + span_len + max_context)

    # If excerpt covers most of the source sentence, just use full sentence
    excerpt_len  = right - left
    coverage     = excerpt_len / max(len(src_tokens_orig), 1)
    if coverage >= 0.85:
        return src_sentence

    excerpt_tokens = src_tokens_orig[left:right]
    excerpt        = " ".join(excerpt_tokens)

    # Capitalise first word
    if excerpt:
        excerpt = excerpt[0].upper() + excerpt[1:]
        if not excerpt.endswith("."):
            excerpt += "."

    return excerpt


def inject_verbatim_spans(lcs_fused_summary, doc_annotation,
                           min_span=VERBATIM_MIN_SPAN):
    """
    ✨ NOVEL IDEA 9 — Verbatim Span Injection for ROUGE-L improvement.

    For each sentence in the LCS-fused summary, attempt to find a critical-
    role source sentence that shares a long verbatim contiguous n-gram span.
    If found and gating passes, substitute the generated sentence with a
    verbatim excerpt from the source (or the full source sentence).

    Args:
        lcs_fused_summary: str  — output of lcs_guided_sentence_fusion()
        doc_annotation:    dict — sentence-role annotations for this document
        min_span:          int  — minimum span length to trigger substitution

    Returns:
        injected_summary:  str  — summary with verbatim spans injected
        injection_log:     dict — per-sentence diagnostics
    """
    injection_log = {
        "method":          "verbatim_span_injection (Novel Idea 9)",
        "min_span":        min_span,
        "target_roles":    VERBATIM_TARGET_ROLES,
        "substitutions":   [],
        "total_sentences": 0,
        "substituted":     0,
        "kept_original":   0
    }

    # Collect critical source sentences for each target role
    # We index them as (role, original_text, lowercase_tokens, original_tokens)
    critical_source = []
    for s in doc_annotation["sentences"]:
        if s["role"] in VERBATIM_TARGET_ROLES:
            orig_tokens  = word_tokenize(s["sentence"])
            lower_tokens = [t.lower() for t in orig_tokens]
            critical_source.append({
                "sentence":     s["sentence"],
                "role":         s["role"],
                "orig_tokens":  orig_tokens,
                "lower_tokens": lower_tokens
            })

    if not critical_source:
        injection_log["early_exit"] = "No critical source sentences found"
        return lcs_fused_summary, injection_log

    # Pre-compute LCS pool for gating comparisons
    critical_pool_text = [s["sentence"] for s in critical_source]

    summary_sentences = sent_tokenize(lcs_fused_summary)
    injection_log["total_sentences"] = len(summary_sentences)
    result_sents      = []

    for sent_idx, summ_sent in enumerate(summary_sentences):
        sent_tokens_lower = word_tokenize(summ_sent.lower())
        sent_tokens_orig  = word_tokenize(summ_sent)

        if not sent_tokens_lower:
            result_sents.append(summ_sent)
            injection_log["kept_original"] += 1
            continue

        # ── Find best verbatim match across all critical source sentences ──
        best_span_len  = 0
        best_src_sent  = None
        best_src_start = -1

        for src_item in critical_source:
            span_len, src_start, _ = _find_longest_verbatim_span(
                sent_tokens_lower, src_item["lower_tokens"], min_span
            )
            if span_len > best_span_len:
                best_span_len  = span_len
                best_src_sent  = src_item
                best_src_start = src_start

        # ── Gating checks ──────────────────────────────────────────────────
        if best_span_len < min_span or best_src_sent is None:
            result_sents.append(summ_sent)
            injection_log["kept_original"] += 1
            continue

        src_tokens_lower = best_src_sent["lower_tokens"]
        src_tokens_orig  = best_src_sent["orig_tokens"]
        src_sentence     = best_src_sent["sentence"]

        # Gate 1: Source sentence must not be too much longer than generated
        len_ratio = len(src_tokens_lower) / max(len(sent_tokens_lower), 1)
        if len_ratio > VERBATIM_MAX_LENGTH_RATIO:
            # Source is too long — try excerpt substitution instead
            excerpt = _extract_verbatim_excerpt(
                src_sentence, src_tokens_orig,
                best_src_start, best_span_len,
                len(sent_tokens_lower)
            )
            excerpt_tokens = word_tokenize(excerpt.lower())
            excerpt_ratio  = len(excerpt_tokens) / max(len(sent_tokens_lower), 1)
            if excerpt_ratio > VERBATIM_MAX_LENGTH_RATIO:
                # Even excerpt is too long — keep original
                result_sents.append(summ_sent)
                injection_log["kept_original"] += 1
                continue
            candidate_text   = excerpt
            candidate_tokens = excerpt_tokens
            substitution_type = "excerpt"
        else:
            candidate_text   = src_sentence
            candidate_tokens = src_tokens_lower
            substitution_type = "full_sentence"

        # Gate 2: Candidate must have higher LCS against critical pool
        # than the original generated sentence
        orig_lcs_score = compute_lcs_score(summ_sent, critical_pool_text)
        cand_lcs_score = compute_lcs_score(candidate_text, critical_pool_text)

        if cand_lcs_score <= orig_lcs_score:
            # Substitution does not improve LCS — keep original
            result_sents.append(summ_sent)
            injection_log["kept_original"] += 1
            continue

        # ── Substitution accepted ──────────────────────────────────────────
        result_sents.append(candidate_text)
        injection_log["substituted"] += 1

        if len(injection_log["substitutions"]) < VERBATIM_LOG_LIMIT:
            injection_log["substitutions"].append({
                "sent_idx":         sent_idx,
                "original":         summ_sent[:100],
                "substituted_with": candidate_text[:100],
                "span_len":         best_span_len,
                "role":             best_src_sent["role"],
                "type":             substitution_type,
                "orig_lcs":         round(orig_lcs_score, 4),
                "new_lcs":          round(cand_lcs_score, 4),
                "lcs_gain":         round(cand_lcs_score - orig_lcs_score, 4)
            })

    injected_summary = " ".join(result_sents)
    injection_log["substitution_rate"] = round(
        injection_log["substituted"] / max(injection_log["total_sentences"], 1), 4
    )
    return injected_summary, injection_log


# =========================================================
# ✨ NOVEL IDEA 10: KG-ATTENTION DURING GENERATION
# ─────────────────────────────────────────────────────────
#
# IMPLEMENTATION DETAILS:
#
#   KGEntityLogitsBoost is a HuggingFace LogitsProcessor.
#   It is called at every decoding step BEFORE the softmax.
#
#   ENTITY TOKEN BUILDING:
#     Each entity string (from KG triple subjects/objects) is tokenized
#     using the target model's tokenizer. We collect all token IDs that
#     appear in ANY entity string. Multi-token entities get a higher boost
#     (ENTITY_BASE_BOOST * ENTITY_MULTI_TOKEN_MULTIPLIER) because they are
#     harder to generate and more discriminative for ROUGE matching.
#
#   DECAY:
#     For each decoding step, we check the already-generated token sequence
#     (input_ids[:, :current_position]) for occurrences of each entity token.
#     Each occurrence reduces the boost by a factor of ENTITY_DECAY_FACTOR.
#     Formula: effective_boost = base_boost * (ENTITY_DECAY_FACTOR ^ count)
#     This allows the entity to appear once (high boost) but discourages
#     repetition (decaying boost).
#
#   CLIPPING:
#     Effective boost is clamped to [0, ENTITY_MAX_BOOST] before adding
#     to logits. This prevents numerical overflow and degenerate outputs.
#
#   THREAD SAFETY:
#     Each call to generate_summary_with_kg_attention() creates a new
#     KGEntityLogitsBoost instance, so there is no shared state between
#     calls. Safe for sequential (single-GPU) inference.
# =========================================================

def extract_critical_entities_from_doc(doc_annotation,
                                        critical_roles=KG_CRITICAL_ROLES):
    """
    Extract entity strings from KG triples of critical-role sentences.
    Returns a list of entity strings (subjects + objects from triples).

    These are the exact legal entity strings we want the generation model
    to produce verbatim, boosted via KG-Attention.

    Args:
        doc_annotation: dict — sentence-role annotations
        critical_roles: list — roles to extract entities from

    Returns:
        List[str] — deduplicated entity strings, sorted by length (longest first)
    """
    critical_sents = [
        s["sentence"] for s in doc_annotation["sentences"]
        if s["role"] in critical_roles
    ]
    if not critical_sents:
        return []

    triples = extract_triples_as_tuples(critical_sents)

    entities = set()
    for subj, rel, obj in triples:
        # Include multi-word entities and single words ≥ ENTITY_MIN_TOKEN_LEN tokens
        for entity in [subj, obj]:
            entity = entity.strip()
            if len(entity.split()) >= ENTITY_MIN_TOKEN_LEN:
                entities.add(entity)
            # Also include individual words from multi-word entities if ≥ 4 chars
            for word in entity.split():
                if len(word) >= 4:
                    entities.add(word)

    # Sort by length descending — longer entities processed first to ensure
    # multi-token boost is computed before individual token boost
    return sorted(entities, key=len, reverse=True)


class KGEntityLogitsBoost(LogitsProcessor):
    """
    ✨ Novel Idea 10 — KG-Attention LogitsProcessor.

    Adds an additive boost to logits of tokens belonging to critical KG
    entities at every decoding step. Boost decays per occurrence in the
    already-generated prefix to prevent repetition.

    Args:
        entity_strings:  List[str] — entity strings from KG triples
        tokenizer:       PreTrainedTokenizer — model's tokenizer
        base_boost:      float — base additive logit boost
        multi_mult:      float — multiplier for tokens in multi-word entities
        decay_factor:    float — boost decay per occurrence in prefix
        max_boost:       float — clamp on effective boost
    """

    def __init__(self, entity_strings, tokenizer,
                 base_boost=ENTITY_BASE_BOOST,
                 multi_mult=ENTITY_MULTI_TOKEN_MULTIPLIER,
                 decay_factor=ENTITY_DECAY_FACTOR,
                 max_boost=ENTITY_MAX_BOOST):
        self.decay_factor = decay_factor
        self.max_boost    = max_boost

        # Build token_id → boost_score mapping
        # Multi-word entities get higher boost per token
        self.token_boost = {}  # token_id (int) → base boost (float)

        for entity in entity_strings:
            entity_token_ids = tokenizer.encode(
                entity, add_special_tokens=False
            )
            is_multi = len(entity_token_ids) > 1
            boost    = base_boost * (multi_mult if is_multi else 1.0)
            for tok_id in entity_token_ids:
                # Take max boost if token appears in multiple entities
                self.token_boost[tok_id] = max(
                    self.token_boost.get(tok_id, 0.0), boost
                )

        # Pre-build sorted list for efficient iteration
        self.boosted_ids    = list(self.token_boost.keys())
        self.boosted_boosts = [self.token_boost[tid] for tid in self.boosted_ids]

        # Convert to tensors for fast indexing
        self.boosted_ids_tensor    = None  # built lazily on first call
        self.boosted_boosts_tensor = None

    def _build_tensors(self, device):
        """Lazily build tensors on the target device."""
        if self.boosted_ids_tensor is None or \
                self.boosted_ids_tensor.device != device:
            self.boosted_ids_tensor = torch.tensor(
                self.boosted_ids, dtype=torch.long, device=device
            )
            self.boosted_boosts_tensor = torch.tensor(
                self.boosted_boosts, dtype=torch.float, device=device
            )

    def __call__(self, input_ids: torch.LongTensor,
                 scores: torch.FloatTensor) -> torch.FloatTensor:
        """
        Called at each decoding step.
        input_ids: (batch, current_seq_len) — already-generated tokens
        scores:    (batch, vocab_size)       — raw logits to be modified
        """
        if not self.boosted_ids:
            return scores

        self._build_tensors(scores.device)

        batch_size = scores.shape[0]

        for b in range(batch_size):
            prefix = input_ids[b]  # already-generated token IDs for this beam

            for i, tok_id in enumerate(self.boosted_ids):
                base   = self.boosted_boosts[i]
                # Count occurrences of this token in the prefix
                count  = (prefix == tok_id).sum().item()
                # Decay the boost per occurrence
                eff    = base * (self.decay_factor ** count)
                # Clamp to [0, max_boost]
                eff    = min(max(eff, 0.0), self.max_boost)
                # Add boost to this token's logit
                scores[b, tok_id] += eff

        return scores


def generate_summary_with_kg_attention(model_name, filtered_text,
                                        doc_annotation):
    """
    ✨ Novel Idea 10 — Generate summary with KG-Attention entity boosting.

    Extracts critical entities from the document's KG triples and applies
    KGEntityLogitsBoost during beam search, steering the model towards
    verbatim use of key legal entity strings.

    Only applied to models in ENTITY_BOOST_MODELS (LED by default).
    Falls back to standard generation for other models.

    Args:
        model_name:     str  — "BART", "LED", or "PEGASUS"
        filtered_text:  str  — hybrid-selected input text
        doc_annotation: dict — sentence-role annotations for this document

    Returns:
        str — generated summary (with entity boosting if applicable)
    """
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]

    inputs = tokenizer(
        filtered_text, return_tensors="pt",
        truncation=True, max_length=config["max_input"]
    ).to(device)

    # ── Build KG-Attention processor ──────────────────────────────────────
    if model_name in ENTITY_BOOST_MODELS:
        entity_strings = extract_critical_entities_from_doc(doc_annotation)

        if entity_strings:
            kg_processor    = KGEntityLogitsBoost(
                entity_strings=entity_strings,
                tokenizer=tokenizer,
                base_boost=ENTITY_BASE_BOOST,
                multi_mult=ENTITY_MULTI_TOKEN_MULTIPLIER,
                decay_factor=ENTITY_DECAY_FACTOR,
                max_boost=ENTITY_MAX_BOOST
            )
            logits_processor = LogitsProcessorList([kg_processor])
        else:
            logits_processor = LogitsProcessorList([])
    else:
        logits_processor = LogitsProcessorList([])

    # ── Generate ──────────────────────────────────────────────────────────
    with torch.no_grad():
        if model_name == "LED":
            global_attention_mask = torch.zeros_like(inputs["attention_mask"])
            global_attention_mask[:, 0] = 1
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=global_attention_mask,
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                logits_processor=logits_processor
            )
        else:
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                logits_processor=logits_processor
            )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# =========================================================
# NOVEL IDEA 5 + 7: KG-DIFF ITERATIVE REFINEMENT
# =========================================================

def kg_iterative_refinement(summaries_dict, doc_annotation,
                             max_iterations=KG_MAX_ITERATIONS):
    """KG-Diff Iterative Refinement with Semantic KG Coverage."""
    refinement_log = {
        "method":             "kg_diff_iterative_refinement_semantic_coverage",
        "novel_ideas":        ["5 (KG-Diff loop)", "7 (Semantic coverage)"],
        "starting_model":     "LED",
        "correction_model":   "PEGASUS",
        "critical_roles":     KG_CRITICAL_ROLES,
        "semantic_threshold": KG_SEMANTIC_THRESHOLD,
        "coverage_threshold": KG_COVERAGE_THRESHOLD,
        "iterations":         []
    }

    critical_sentences = [
        s["sentence"] for s in doc_annotation["sentences"]
        if s["role"] in KG_CRITICAL_ROLES
    ]

    if not critical_sentences:
        refinement_log["early_exit"] = "No critical sentences found"
        return summaries_dict.get("LED", ""), refinement_log

    critical_triples = extract_triples_as_tuples(critical_sentences)
    refinement_log["source_critical_triples_count"] = len(critical_triples)

    if not critical_triples:
        refinement_log["early_exit"] = "No triples extracted"
        return summaries_dict.get("LED", ""), refinement_log

    # Use the KG-Attention LED output as the starting point (Novel Idea 10)
    current_summary = (
        summaries_dict.get("LED_kg_attn", "") or
        summaries_dict.get("LED", "") or
        summaries_dict.get("PEGASUS", "")
    )

    for iteration in range(max_iterations):
        summary_sentences = sent_tokenize(current_summary)
        summary_triples   = extract_triples_as_tuples(summary_sentences)

        coverage, per_triple_scores, uncovered_triples = compute_semantic_kg_coverage(
            critical_triples, summary_triples, KG_SEMANTIC_THRESHOLD
        )

        iter_log = {
            "iteration":              iteration + 1,
            "coverage_method":        "semantic_cosine (Novel Idea 7)",
            "semantic_threshold":     KG_SEMANTIC_THRESHOLD,
            "summary_triples_count":  len(summary_triples),
            "critical_triples_count": len(critical_triples),
            "uncovered_count":        len(uncovered_triples),
            "coverage_before":        round(coverage, 4),
            "worst_uncovered_sample": [
                {"triple": triple_to_text(t), "max_sim": round(sim, 4)}
                for t, sim in uncovered_triples[:3]
            ]
        }

        if coverage >= KG_COVERAGE_THRESHOLD:
            iter_log["action"] = (
                f"STOPPED — coverage {coverage:.3f} >= {KG_COVERAGE_THRESHOLD}"
            )
            refinement_log["iterations"].append(iter_log)
            break

        if not uncovered_triples:
            iter_log["action"] = "STOPPED — all triples covered"
            refinement_log["iterations"].append(iter_log)
            break

        missing_entities = get_missing_entities_from_uncovered(
            uncovered_triples, top_k=KG_TOP_MISSING
        )
        iter_log["missing_entities_sample"] = list(missing_entities)[:10]

        correction_sentences = []
        for s in doc_annotation["sentences"]:
            sent_text  = s["sentence"]
            sent_lower = sent_text.lower()
            role       = s["role"]
            if any(ent in sent_lower for ent in missing_entities):
                priority = 0 if role in KG_CRITICAL_ROLES else 1
                correction_sentences.append((priority, sent_text))

        correction_sentences.sort(key=lambda x: x[0])
        correction_sentences = [
            s for _, s in correction_sentences[:KG_MAX_CORRECTION_SENTS]
        ]

        if not correction_sentences:
            iter_log["action"] = "STOPPED — no correction sentences"
            refinement_log["iterations"].append(iter_log)
            break

        iter_log["correction_sentences_count"] = len(correction_sentences)
        correction_context  = " ".join(correction_sentences)
        correction_input    = (
            f"Improve this legal summary by including the missing information.\n\n"
            f"Current summary: {current_summary}\n\n"
            f"Missing information: {correction_context}\n\n"
            f"Improved summary:"
        )
        iter_log["correction_input_chars"] = len(correction_input)

        refined_summary  = generate_summary("PEGASUS", correction_input)
        refined_triples  = extract_triples_as_tuples(sent_tokenize(refined_summary))
        refined_coverage, _, _ = compute_semantic_kg_coverage(
            critical_triples, refined_triples, KG_SEMANTIC_THRESHOLD
        )

        iter_log["coverage_after"] = round(refined_coverage, 4)
        iter_log["coverage_delta"] = round(refined_coverage - coverage, 4)

        if refined_coverage > coverage:
            current_summary = refined_summary
            iter_log["action"] = f"ACCEPTED — {coverage:.3f} → {refined_coverage:.3f}"
        else:
            iter_log["action"] = f"REJECTED — {refined_coverage:.3f} <= {coverage:.3f}"

        refinement_log["iterations"].append(iter_log)

    final_triples = extract_triples_as_tuples(sent_tokenize(current_summary))
    final_coverage, final_per_triple, _ = compute_semantic_kg_coverage(
        critical_triples, final_triples, KG_SEMANTIC_THRESHOLD
    )
    refinement_log["final_semantic_coverage"]  = round(final_coverage, 4)
    refinement_log["total_iterations_run"]     = len(refinement_log["iterations"])
    refinement_log["final_coverage_breakdown"] = {
        "covered":   sum(1 for _, _, c in final_per_triple if c),
        "uncovered": sum(1 for _, _, c in final_per_triple if not c),
        "total":     len(final_per_triple)
    }
    return current_summary, refinement_log


# =========================================================
# STEP 1: CREATE SENTENCE-ROLE ANNOTATION FILE (8 LABELS)
# =========================================================

def create_role_annotations(data, output_file):
    """Create intermediate file with sentence-role mappings (8 labels)."""
    print("\n" + "="*70)
    print("STEP 1: CREATING SENTENCE-ROLE ANNOTATIONS (8 LABELS)")
    print("="*70)

    annotations = []
    for idx, item in enumerate(tqdm(data, desc="Annotating documents")):
        judgment  = item.get("judgment", "")
        doc_id    = item.get("id", idx)
        sentences = sent_tokenize(judgment)
        if not sentences:
            continue
        roles_8 = classify_roles(sentences)
        doc_annotation = {
            "doc_id":          doc_id,
            "total_sentences": len(sentences),
            "sentences":       []
        }
        for sent_idx, (sentence, role) in enumerate(zip(sentences, roles_8)):
            doc_annotation["sentences"].append({
                "index":    sent_idx,
                "sentence": sentence,
                "role":     role
            })
        annotations.append(doc_annotation)

    with open(output_file, 'w', encoding='utf8') as f:
        json.dump(annotations, f, indent=2, ensure_ascii=False)

    print(f"✅ Annotations saved to: {output_file}")
    print(f"   Total documents annotated: {len(annotations)}")
    print_role_statistics(annotations)
    return annotations


def print_role_statistics(annotations):
    role_counts     = Counter()
    total_sentences = 0
    for doc in annotations:
        for sent in doc["sentences"]:
            role_counts[sent["role"]] += 1
            total_sentences += 1
    print("\n📊 Role Distribution (8 Labels):")
    print("-" * 60)
    for role in LABELS_8:
        count = role_counts[role]
        pct   = (count / total_sentences * 100) if total_sentences > 0 else 0
        bar   = "█" * int(pct / 2)
        print(f"  {role:25s}: {count:6d} ({pct:5.2f}%) {bar}")
    print("-" * 60)
    print(f"  {'TOTAL':25s}: {total_sentences:6d}")
    print("-" * 60)


# =========================================================
# STEP 2: CALCULATE ROLE CONTRIBUTION SCORES (8 LABELS)
# =========================================================

def calculate_role_contribution(train_annotations, train_data, output_file):
    """Calculate role-level contribution scores from training data."""
    print("\n" + "="*70)
    print("STEP 2: CALCULATING ROLE CONTRIBUTION SCORES (8 LABELS)")
    print("="*70)

    role_total_counts   = Counter()
    role_summary_counts = Counter()

    for doc_annotation, train_item in tqdm(
        zip(train_annotations, train_data),
        total=len(train_annotations),
        desc="Computing contributions"
    ):
        reference_summary = train_item.get("reference_summary", "")
        summary_sentences = sent_tokenize(reference_summary)
        doc_sentences     = [s["sentence"] for s in doc_annotation["sentences"]]
        doc_roles         = [s["role"]     for s in doc_annotation["sentences"]]

        for role in doc_roles:
            role_total_counts[role] += 1

        if doc_sentences and summary_sentences:
            doc_embeddings = embed_with_legalbert(doc_sentences)
            sum_embeddings = embed_with_legalbert(summary_sentences)
            for sum_emb in sum_embeddings:
                similarities   = cosine_similarity([sum_emb], doc_embeddings)[0]
                best_match_idx = np.argmax(similarities)
                if similarities[best_match_idx] > 0.7:
                    role_summary_counts[doc_roles[best_match_idx]] += 1

    role_scores = {}
    for role in LABELS_8:
        C_r       = role_total_counts[role]
        sum_alpha = role_summary_counts[role]
        role_scores[role] = sum_alpha / C_r if C_r > 0 else 0.0

    contribution_data = {
        "label_system":        "8 labels",
        "role_scores":         role_scores,
        "role_total_counts":   dict(role_total_counts),
        "role_summary_counts": dict(role_summary_counts),
        "formula":             "RoleScore(r) = (1/C_r) * Σ α_i",
        "mapping_used":        LABEL_MAPPING_13_TO_8
    }
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump(contribution_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Role contribution scores saved to: {output_file}")
    print_contribution_scores(role_scores, role_total_counts, role_summary_counts)
    return role_scores


def print_contribution_scores(role_scores, total_counts, summary_counts):
    print("\n📊 Role Contribution Scores (8 Labels):")
    print("-" * 90)
    print(f"{'Role':<25} {'Total Count':<15} {'In Summaries':<15} {'Score':<10} {'Bar'}")
    print("-" * 90)
    for role, score in sorted(role_scores.items(), key=lambda x: x[1], reverse=True):
        bar = "█" * int(score * 50)
        print(f"{role:<25} {total_counts[role]:<15} {summary_counts[role]:<15} "
              f"{score:<10.4f} {bar}")
    print("-" * 90)


# =========================================================
# STEP 3: NORMALIZE ROLE WEIGHTS (8 LABELS)
# =========================================================

def normalize_role_weights(role_scores, output_file):
    """Normalize role scores: w_r = RoleScore(r) / Σ RoleScore(r)"""
    print("\n" + "="*70)
    print("STEP 3: NORMALIZING ROLE WEIGHTS (8 LABELS)")
    print("="*70)

    total_score = sum(role_scores.values())
    if total_score == 0:
        print("⚠️  Warning: Total score is 0, using uniform weights")
        normalized_weights = {role: 1.0 / len(LABELS_8) for role in LABELS_8}
    else:
        normalized_weights = {r: s / total_score for r, s in role_scores.items()}

    weights_data = {
        "label_system":       "8 labels",
        "normalized_weights": normalized_weights,
        "original_scores":    role_scores,
        "formula":            "w_r = RoleScore(r) / Σ RoleScore(r)",
        "mapping_used":       LABEL_MAPPING_13_TO_8
    }
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump(weights_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Normalized weights saved to: {output_file}")
    print_normalized_weights(normalized_weights)
    return normalized_weights


def print_normalized_weights(weights):
    print("\n📊 Normalized Role Weights (8 Labels):")
    print("-" * 75)
    print(f"{'Role':<25} {'Weight':<10} {'Percentage':<12} {'Bar'}")
    print("-" * 75)
    for role, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        pct = weight * 100
        bar = "█" * int(pct * 2)
        print(f"{role:<25} {weight:<10.4f} {pct:<12.2f}% {bar}")
    print("-" * 75)
    total = sum(weights.values())
    print(f"{'SUM':<25} {total:<10.4f} {total*100:.2f}%")
    print("-" * 75)


# =========================================================
# HYBRID ROLE-SALIENCE SELECTION
# =========================================================

def hybrid_role_salience_selection(doc_annotation, normalized_weights,
                                    target_sentences):
    """Hybrid scoring combining role importance with sentence salience."""
    sentences = [s["sentence"] for s in doc_annotation["sentences"]]
    roles     = [s["role"]     for s in doc_annotation["sentences"]]

    if not sentences:
        return "", {"total_available": 0, "selected": 0, "method": "hybrid"}

    embeddings   = embed_with_legalbert(sentences)
    centroid     = embeddings.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(embeddings, centroid).squeeze()

    hybrid_scores = []
    for idx, (role, sim) in enumerate(zip(roles, similarities)):
        role_weight  = normalized_weights.get(role, 0)
        hybrid_score = (HYBRID_ALPHA * role_weight * 10) + (HYBRID_BETA * float(sim))
        hybrid_scores.append({
            "index":       idx,
            "score":       hybrid_score,
            "role":        role,
            "role_weight": role_weight,
            "similarity":  float(sim),
            "sentence":    sentences[idx]
        })

    sorted_sents       = sorted(hybrid_scores, key=lambda x: x["score"], reverse=True)
    selected_items     = sorted_sents[:target_sentences]
    selected_indices   = sorted([s["index"] for s in selected_items])
    selected_sentences = [sentences[i] for i in selected_indices]

    selection_info = {
        "total_available": len(sentences),
        "target":   target_sentences,
        "selected": len(selected_indices),
        "method":   "hybrid_role_salience",
        "formula":  f"{HYBRID_ALPHA} * role_weight * 10 + {HYBRID_BETA} * salience",
        "alpha":    HYBRID_ALPHA,
        "beta":     HYBRID_BETA,
        "role_distribution": dict(Counter([s["role"] for s in selected_items]))
    }
    return " ".join(selected_sentences), selection_info


# =========================================================
# LOAD SUMMARIZATION MODELS
# =========================================================

print("\n📚 Loading summarization models...")
models     = {}
tokenizers = {}

print("\n  [1/3] Loading BART...")
tokenizers["BART"] = AutoTokenizer.from_pretrained(MODEL_PATHS["BART"])
models["BART"]     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATHS["BART"]).to(device)
models["BART"].eval()
print("  ✅ BART loaded")

print("\n  [2/3] Loading LED...")
tokenizers["LED"] = AutoTokenizer.from_pretrained(MODEL_PATHS["LED"])
models["LED"]     = LEDForConditionalGeneration.from_pretrained(MODEL_PATHS["LED"]).to(device)
models["LED"].eval()
print("  ✅ LED loaded")

print("\n  [3/3] Loading PEGASUS...")
tokenizers["PEGASUS"] = AutoTokenizer.from_pretrained(MODEL_PATHS["PEGASUS"])
models["PEGASUS"]     = PegasusForConditionalGeneration.from_pretrained(MODEL_PATHS["PEGASUS"]).to(device)
models["PEGASUS"].eval()
print("  ✅ PEGASUS loaded")

print("\n✅ All models loaded successfully!")


# =========================================================
# STANDARD SUMMARY GENERATION (no KG-Attention)
# =========================================================

def generate_summary(model_name, filtered_text):
    """Generate summary using the specified model (standard beam search)."""
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]

    inputs = tokenizer(
        filtered_text, return_tensors="pt",
        truncation=True, max_length=config["max_input"]
    ).to(device)

    with torch.no_grad():
        if model_name == "LED":
            global_attention_mask = torch.zeros_like(inputs["attention_mask"])
            global_attention_mask[:, 0] = 1
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=global_attention_mask,
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        else:
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3
            )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# =========================================================
# ENSEMBLE STRATEGIES
# =========================================================

def ensemble_voting(summaries_dict):
    """Keep sentences appearing in at least 2 of 3 model summaries."""
    all_sentences   = []
    for summary in summaries_dict.values():
        all_sentences.extend(sent_tokenize(summary))
    sentence_counts = Counter(sent.lower().strip() for sent in all_sentences)
    selected        = [s for s, c in sentence_counts.items() if c >= 2]
    if len(selected) < 3:
        selected = [s for s, _ in sentence_counts.most_common(10)]
    return " ".join(selected)


def ensemble_weighted_concat(summaries_dict, weights):
    """Take proportionally more sentences from higher-weight models."""
    weighted_parts = []
    for model_name, summary in summaries_dict.items():
        weight  = weights[model_name]
        sents   = sent_tokenize(summary)
        n_sents = max(1, int(len(sents) * weight))
        weighted_parts.extend(sents[:n_sents])
    seen = set()
    unique_sents = []
    for sent in weighted_parts:
        norm = sent.lower().strip()
        if norm not in seen:
            seen.add(norm)
            unique_sents.append(sent)
    return " ".join(unique_sents)


def ensemble_best_model(summaries_dict, reference):
    """Best-model selection using ROUGE-2 (uses reference — for analysis only)."""
    best_score, best_summary = -1, ""
    for model_name, summary in summaries_dict.items():
        score = rouge_metric.compute(
            predictions=[summary], references=[reference], use_stemmer=True
        )["rouge2"]
        if score > best_score:
            best_score   = score
            best_summary = summary
    return best_summary


def ensemble_sentence_ranking(summaries_dict):
    """Rank sentences by average position across all summaries, take top 15."""
    sentence_positions = {}
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for pos, sent in enumerate(sents):
            norm = sent.lower().strip()
            if norm not in sentence_positions:
                sentence_positions[norm] = []
            sentence_positions[norm].append(pos)
    sentence_scores  = {
        sent: np.mean(positions)
        for sent, positions in sentence_positions.items()
    }
    ranked        = sorted(sentence_scores.items(), key=lambda x: x[1])
    top_sentences = [sent for sent, _ in ranked[:15]]
    return " ".join(top_sentences)


# =========================================================
# EVALUATION
# =========================================================

print("\n📊 Loading evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")


def compute_metrics(predictions, references):
    """Compute ROUGE-1/2/L and BERTScore."""
    rouge_scores = rouge_metric.compute(
        predictions=predictions, references=references, use_stemmer=True
    )
    bert_scores = bertscore_metric.compute(
        predictions=predictions, references=references,
        model_type="roberta-large", lang="en",
        device=device, batch_size=8
    )
    return {
        "rouge1":       float(rouge_scores["rouge1"]),
        "rouge2":       float(rouge_scores["rouge2"]),
        "rougeL":       float(rouge_scores["rougeL"]),
        "bertscore_f1": float(np.mean(bert_scores["f1"]))
    }


# =========================================================
# MAIN PIPELINE
# =========================================================

def main():
    print("\n" + "="*70)
    print("🚀 ENSEMBLE SUMMARIZATION v2 — ROUGE-L IMPROVEMENT")
    print("   Novel Idea 5:  KG-Diff self-correcting loop")
    print("   Novel Idea 7:  Semantic KG Coverage")
    print("   Novel Idea 8:  LCS-Guided Sentence Fusion")
    print("   ✨ Novel Idea 9:  Verbatim Span Injection")
    print("   ✨ Novel Idea 10: KG-Attention During Generation")
    print("="*70)

    # ── Load data ──────────────────────────────────────────────────────────
    print(f"\n📂 Loading training data from {TRAIN_JSON_PATH}...")
    with open(TRAIN_JSON_PATH, 'r', encoding='utf8') as f:
        train_data = json.load(f)
    train_data = train_data[:MAX_TRAIN_SAMPLES]
    print(f"   Loaded {len(train_data)} training samples")

    print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")
    with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
        test_data = json.load(f)
    print(f"   Loaded {len(test_data)} test samples")

    # ── STEP 1: Role annotations ───────────────────────────────────────────
    role_annotation_path = os.path.join(OUTPUT_DIR, ROLE_CLASSIFICATION_FILE)
    if os.path.exists(role_annotation_path):
        print(f"\n📂 Loading existing role annotations from {role_annotation_path}")
        with open(role_annotation_path, 'r', encoding='utf8') as f:
            train_annotations = json.load(f)
    else:
        train_annotations = create_role_annotations(train_data, role_annotation_path)

    # ── STEP 2: Role contribution scores ──────────────────────────────────
    contribution_path = os.path.join(OUTPUT_DIR, ROLE_CONTRIBUTION_FILE)
    if os.path.exists(contribution_path):
        print(f"\n📂 Loading existing contribution scores from {contribution_path}")
        with open(contribution_path, 'r', encoding='utf8') as f:
            contribution_data = json.load(f)
        role_scores = contribution_data["role_scores"]
        print_contribution_scores(
            role_scores,
            contribution_data["role_total_counts"],
            contribution_data["role_summary_counts"]
        )
    else:
        role_scores = calculate_role_contribution(
            train_annotations, train_data, contribution_path
        )

    # ── STEP 3: Normalize role weights ────────────────────────────────────
    weights_path = os.path.join(OUTPUT_DIR, ROLE_WEIGHTS_FILE)
    if os.path.exists(weights_path):
        print(f"\n📂 Loading existing normalized weights from {weights_path}")
        with open(weights_path, 'r', encoding='utf8') as f:
            weights_data = json.load(f)
        normalized_weights = weights_data["normalized_weights"]
        print_normalized_weights(normalized_weights)
    else:
        normalized_weights = normalize_role_weights(role_scores, weights_path)

    # ── STEP 4: Annotate test documents ───────────────────────────────────
    test_annotation_path = os.path.join(OUTPUT_DIR, "test_" + ROLE_CLASSIFICATION_FILE)
    if os.path.exists(test_annotation_path):
        print(f"\n📂 Loading test annotations from {test_annotation_path}")
        with open(test_annotation_path, 'r', encoding='utf8') as f:
            test_annotations = json.load(f)
    else:
        test_annotations = create_role_annotations(test_data, test_annotation_path)

    # ── STEP 5: Generate summaries ─────────────────────────────────────────
    print("\n" + "="*70)
    print("STEP 5: GENERATING SUMMARIES")
    print("        Pipeline: KG-Attention Generation → KG-Diff Refinement")
    print("                  → LCS Fusion → Verbatim Span Injection")
    print("="*70)

    all_model_summaries = {
        "BART":    [],
        "LED":     [],
        "PEGASUS": [],
        "LED_kg_attn": []   # LED with KG-Attention (Novel Idea 10)
    }

    ensemble_summaries = {
        "voting":           [],
        "weighted":         [],
        "best_model":       [],
        "ranking":          [],
        "kg_refined":       [],   # Novel Ideas 5+7
        "lcs_fused":        [],   # Novel Idea 8
        "verbatim_injected": [],  # ✨ Novel Idea 9: Verbatim Span Injection
        "kg_attn_refined":  []    # ✨ Novel Idea 10: KG-Attention + full pipeline
    }

    references            = []
    all_adaptive_targets  = []
    all_refinement_logs   = []
    all_fusion_logs       = []
    all_injection_logs    = []

    print("\n🔄 Generating summaries...")
    for test_annotation, test_item in tqdm(
        zip(test_annotations, test_data),
        total=len(test_data),
        desc="Processing"
    ):
        reference = test_item["reference_summary"]
        references.append(reference)

        doc_length       = test_annotation["total_sentences"]
        adaptive_targets = get_adaptive_targets(doc_length)
        all_adaptive_targets.append({
            "doc_id":     test_annotation["doc_id"],
            "doc_length": doc_length,
            "targets":    adaptive_targets
        })

        summaries_dict = {}

        # ── Standard generation: BART + PEGASUS ───────────────────────────
        for model_name in ["BART", "PEGASUS"]:
            target        = adaptive_targets[model_name]
            filtered_text, _ = hybrid_role_salience_selection(
                test_annotation, normalized_weights, target
            )
            summary = generate_summary(model_name, filtered_text)
            all_model_summaries[model_name].append(summary)
            summaries_dict[model_name] = summary

        # ── Standard LED generation ────────────────────────────────────────
        led_target       = adaptive_targets["LED"]
        led_filtered, _  = hybrid_role_salience_selection(
            test_annotation, normalized_weights, led_target
        )
        led_summary_std  = generate_summary("LED", led_filtered)
        all_model_summaries["LED"].append(led_summary_std)
        summaries_dict["LED"] = led_summary_std

        # ── ✨ Novel Idea 10: LED with KG-Attention ────────────────────────
        led_summary_kg_attn = generate_summary_with_kg_attention(
            "LED", led_filtered, test_annotation
        )
        all_model_summaries["LED_kg_attn"].append(led_summary_kg_attn)
        summaries_dict["LED_kg_attn"] = led_summary_kg_attn

        # ── Original ensemble strategies (using standard LED) ──────────────
        std_summaries = {
            "BART":    summaries_dict["BART"],
            "LED":     summaries_dict["LED"],
            "PEGASUS": summaries_dict["PEGASUS"]
        }
        ensemble_summaries["voting"].append(ensemble_voting(std_summaries))
        ensemble_summaries["weighted"].append(
            ensemble_weighted_concat(std_summaries, ENSEMBLE_WEIGHTS)
        )
        ensemble_summaries["best_model"].append(
            ensemble_best_model(std_summaries, reference)
        )
        ensemble_summaries["ranking"].append(
            ensemble_sentence_ranking(std_summaries)
        )

        # ── Novel Ideas 5+7: KG-Diff (using standard LED) ─────────────────
        kg_refined_summary, refinement_log = kg_iterative_refinement(
            summaries_dict, test_annotation, max_iterations=KG_MAX_ITERATIONS
        )
        ensemble_summaries["kg_refined"].append(kg_refined_summary)

        if len(all_refinement_logs) < 3:
            all_refinement_logs.append({
                "doc_id": test_annotation["doc_id"],
                "log":    refinement_log
            })

        # ── Novel Idea 8: LCS-Guided Fusion ───────────────────────────────
        lcs_fused_summary, fusion_log = lcs_guided_sentence_fusion(
            kg_refined_summary, test_annotation, normalized_weights
        )
        ensemble_summaries["lcs_fused"].append(lcs_fused_summary)

        if len(all_fusion_logs) < 3:
            all_fusion_logs.append({
                "doc_id": test_annotation["doc_id"],
                "log":    fusion_log
            })

        # ── ✨ Novel Idea 9: Verbatim Span Injection ───────────────────────
        # Applied on top of LCS-fused output (stacked improvement)
        verbatim_summary, injection_log = inject_verbatim_spans(
            lcs_fused_summary, test_annotation, min_span=VERBATIM_MIN_SPAN
        )
        ensemble_summaries["verbatim_injected"].append(verbatim_summary)

        if len(all_injection_logs) < 3:
            all_injection_logs.append({
                "doc_id": test_annotation["doc_id"],
                "log":    injection_log
            })

        # ── ✨ Novel Idea 10 full pipeline: KG-Attn → KG-Diff → LCS → Verbatim
        # Uses the KG-Attention LED output as the foundation, then applies
        # the full post-processing stack for maximum ROUGE-L gain.
        kg_attn_summaries_dict = {
            "BART":      summaries_dict["BART"],
            "LED":       summaries_dict["LED_kg_attn"],  # ← KG-Attention LED
            "PEGASUS":   summaries_dict["PEGASUS"],
            "LED_kg_attn": summaries_dict["LED_kg_attn"]
        }
        kg_attn_refined, _ = kg_iterative_refinement(
            kg_attn_summaries_dict, test_annotation,
            max_iterations=KG_MAX_ITERATIONS
        )
        kg_attn_fused, _   = lcs_guided_sentence_fusion(
            kg_attn_refined, test_annotation, normalized_weights
        )
        kg_attn_verbatim, _ = inject_verbatim_spans(
            kg_attn_fused, test_annotation, min_span=VERBATIM_MIN_SPAN
        )
        ensemble_summaries["kg_attn_refined"].append(kg_attn_verbatim)

    print("✅ All summaries generated!")

    # ── Save metadata ──────────────────────────────────────────────────────
    meta_files = {
        "adaptive_targets_used.json":         all_adaptive_targets,
        "semantic_kg_refinement_logs.json":   all_refinement_logs,
        "lcs_fusion_logs.json":               all_fusion_logs,
        "verbatim_injection_logs.json":       all_injection_logs,
    }
    for fname, data in meta_files.items():
        with open(os.path.join(OUTPUT_DIR, fname), 'w', encoding='utf8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"\n📊 Logs saved to: {OUTPUT_DIR}/")

    # ── Evaluate ALL approaches ────────────────────────────────────────────
    print("\n" + "="*70)
    print("📊 EVALUATING ALL APPROACHES")
    print("="*70)

    results = {}

    for model_name in ["BART", "LED", "PEGASUS", "LED_kg_attn"]:
        label = model_name
        if model_name == "LED_kg_attn":
            label = "LED_kg_attn ✨ (Novel 10)"
        print(f"\n  Evaluating {label}...")
        metrics = compute_metrics(all_model_summaries[model_name], references)
        results[model_name] = metrics
        tag = " 🎯 TARGET MET!" if metrics['rougeL'] >= 0.34 else ""
        print(f"  ✅ R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
              f"RL:{metrics['rougeL']:.4f}{tag}  BS:{metrics['bertscore_f1']:.4f}")

    strategy_labels = {
        "voting":            "Ensemble Voting",
        "weighted":          "Ensemble Weighted",
        "best_model":        "Ensemble Best (uses ref)",
        "ranking":           "Ensemble Ranking",
        "kg_refined":        "KG-Refined (Novel 5+7)",
        "lcs_fused":         "LCS-Fused (Novel 8)",
        "verbatim_injected": "Verbatim Injected ✨ (Novel 9)",
        "kg_attn_refined":   "KG-Attn+Full Stack ✨ (Novel 10)"
    }

    for strategy, label in strategy_labels.items():
        print(f"\n  Evaluating {label}...")
        metrics = compute_metrics(ensemble_summaries[strategy], references)
        results[f"ensemble_{strategy}"] = metrics
        tag = " 🎯 TARGET MET!" if metrics['rougeL'] >= 0.34 else ""
        print(f"  ✅ R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
              f"RL:{metrics['rougeL']:.4f}{tag}  BS:{metrics['bertscore_f1']:.4f}")

    # ── Results table ──────────────────────────────────────────────────────
    print("\n" + "="*90)
    print("📊 FINAL RESULTS — All Approaches")
    print("="*90)
    print(f"\n{'Approach':<45} {'R1':<9} {'R2':<9} {'RL':<9} {'Status':<16} {'BertF1'}")
    print("-" * 90)

    rows = []
    for approach, metrics in results.items():
        rl     = metrics['rougeL']
        status = "✓ ≥0.34" if rl >= 0.34 else f"({0.34-rl:.4f} short)"
        star   = " ✨" if approach in ("ensemble_verbatim_injected",
                                       "ensemble_kg_attn_refined",
                                       "LED_kg_attn") else ""
        rows.append((approach, metrics, rl, status, star))

    rows.sort(key=lambda x: x[2], reverse=True)
    for approach, metrics, rl, status, star in rows:
        print(f"{approach+star:<45} {metrics['rouge1']:<9.4f} {metrics['rouge2']:<9.4f} "
              f"{rl:<9.4f} {status:<16} {metrics['bertscore_f1']:.4f}")

    best_rougeL = max(results.items(), key=lambda x: x[1]['rougeL'])
    best_rouge2 = max(results.items(), key=lambda x: x[1]['rouge2'])
    print("\n" + "="*90)
    print(f"🏆 BEST ROUGE-L: {best_rougeL[0]} → {best_rougeL[1]['rougeL']:.4f}")
    print(f"🏆 BEST ROUGE-2: {best_rouge2[0]} → {best_rouge2[1]['rouge2']:.4f}")
    print("="*90)

    # ── Save summaries ─────────────────────────────────────────────────────
    print("\n💾 Saving summaries...")
    idx_map = list(zip(test_data, references))

    for model_name in ["BART", "LED", "PEGASUS", "LED_kg_attn"]:
        output_path = os.path.join(OUTPUT_DIR, f"{model_name.lower()}_summaries.json")
        with open(output_path, 'w', encoding='utf8') as f:
            json.dump([
                {
                    "id":                item.get("id", idx),
                    "generated_summary": summary,
                    "reference_summary": ref
                }
                for idx, ((item, ref), summary) in enumerate(
                    zip(idx_map, all_model_summaries[model_name])
                )
            ], f, indent=2, ensure_ascii=False)

    for strategy in ensemble_summaries:
        output_path = os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json")
        with open(output_path, 'w', encoding='utf8') as f:
            json.dump([
                {
                    "id":                item.get("id", idx),
                    "generated_summary": summary,
                    "reference_summary": ref
                }
                for idx, ((item, ref), summary) in enumerate(
                    zip(idx_map, ensemble_summaries[strategy])
                )
            ], f, indent=2, ensure_ascii=False)

    # ── Save complete results ──────────────────────────────────────────────
    complete_results = {
        "experiment": "Ensemble Summarization v2 — Novel Ideas 9 + 10",
        "novel_idea_9": {
            "name":   "Verbatim Span Injection",
            "target": "ROUGE-L improvement via verbatim span substitution",
            "config": {
                "min_span":           VERBATIM_MIN_SPAN,
                "target_roles":       VERBATIM_TARGET_ROLES,
                "max_length_ratio":   VERBATIM_MAX_LENGTH_RATIO,
                "max_context_tokens": VERBATIM_MAX_CONTEXT_TOKENS
            },
            "builds_on": "lcs_fused (Novel Idea 8)"
        },
        "novel_idea_10": {
            "name":   "KG-Attention During Generation",
            "target": "Better entity coverage + ROUGE-L via boosted beam search",
            "config": {
                "base_boost":      ENTITY_BASE_BOOST,
                "multi_mult":      ENTITY_MULTI_TOKEN_MULTIPLIER,
                "decay_factor":    ENTITY_DECAY_FACTOR,
                "max_boost":       ENTITY_MAX_BOOST,
                "boost_models":    ENTITY_BOOST_MODELS,
                "min_token_len":   ENTITY_MIN_TOKEN_LEN
            },
            "applied_to": "LED only (highest quality baseline)"
        },
        "ensemble_weights": ENSEMBLE_WEIGHTS,
        "test_samples":     len(test_data),
        "results":          results,
        "best_rougeL": {
            "name":    best_rougeL[0],
            "metrics": best_rougeL[1]
        },
        "best_rouge2": {
            "name":    best_rouge2[0],
            "metrics": best_rouge2[1]
        }
    }

    results_path = os.path.join(OUTPUT_DIR, "complete_results_v2.json")
    with open(results_path, 'w', encoding='utf8') as f:
        json.dump(complete_results, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Complete results saved to: {results_path}")
    print("\n" + "="*70)
    print("✅ PIPELINE v2 COMPLETE!")
    print("="*70)
    print("\n💡 Novel Idea 9 — Verbatim Span Injection:")
    print(f"   Min span:    {VERBATIM_MIN_SPAN} tokens")
    print(f"   Roles:       {VERBATIM_TARGET_ROLES}")
    print(f"   Max ratio:   {VERBATIM_MAX_LENGTH_RATIO}x source/generated length")
    print(f"   Context exp: ±{VERBATIM_MAX_CONTEXT_TOKENS} tokens around span")
    print(f"   Gate:        Only substitute if LCS improves")
    print("\n💡 Novel Idea 10 — KG-Attention During Generation:")
    print(f"   Base boost:  {ENTITY_BASE_BOOST}")
    print(f"   Multi mult:  {ENTITY_MULTI_TOKEN_MULTIPLIER}x for multi-word entities")
    print(f"   Decay:       {ENTITY_DECAY_FACTOR}^count per occurrence in prefix")
    print(f"   Cap:         max {ENTITY_MAX_BOOST} per token")
    print(f"   Models:      {ENTITY_BOOST_MODELS}")
    print("\n📊 Full stacked pipeline:")
    print("   KG-Attention LED → KG-Diff (Novel 5+7) → LCS Fusion (Novel 8)")
    print("   → Verbatim Injection (Novel 9)")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Pipeline interrupted by user")
    except Exception as e:
        print(f"\n\n❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

📥 Downloading NLTK data...
🚀 Device: cuda

📋 CONFIGURATION — ROUGE-L IMPROVEMENT v2 (Novel Ideas 9 + 10)
Label System:   8 Strategic Labels
Selection:      Hybrid Role Weight + Content Salience

✨ Novel Idea 9: Verbatim Span Injection
  Min span:            5 tokens
  Target roles:        ['ISSUE', 'REASONING', 'PRECEDENT_RELIED']
  Max length ratio:    2.5x
  Context expansion:   ±10 tokens

✨ Novel Idea 10: KG-Attention During Generation
  Base boost:          1.5
  Multi-token mult:    1.3x
  Decay factor:        0.6 per occurrence
  Max boost cap:       3.0
  Applied to models:   ['LED']

⚡ ENSEMBLE WEIGHTS:
   BART:    0.25
   LED:     0.50 ← BEST baseline
   PEGASUS: 0.25

Output directory: ./ensemble_results_8label_rougeL_v2

📚 Loading HSLN role classifier (13 labels)...
✅ HSLN role classifier loaded (13 labels → mapped to 8)

📚 Loading InLegalBERT for embeddings...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded successfully

📚 Loading summarization models...

  [1/3] Loading BART...
  ✅ BART loaded

  [2/3] Loading LED...
  ✅ LED loaded

  [3/3] Loading PEGASUS...
  ✅ PEGASUS loaded

✅ All models loaded successfully!

📊 Loading evaluation metrics...

🚀 ENSEMBLE SUMMARIZATION v2 — ROUGE-L IMPROVEMENT
   Novel Idea 5:  KG-Diff self-correcting loop
   Novel Idea 7:  Semantic KG Coverage
   Novel Idea 8:  LCS-Guided Sentence Fusion
   ✨ Novel Idea 9:  Verbatim Span Injection
   ✨ Novel Idea 10: KG-Attention During Generation

📂 Loading training data from train.json...
   Loaded 3000 training samples

📂 Loading test data from test.json...
   Loaded 100 test samples

STEP 1: CREATING SENTENCE-ROLE ANNOTATIONS (8 LABELS)


Annotating documents: 100%|█████████████████████████████████████████████████████████| 3000/3000 [10:39<00:00,  4.69it/s]


✅ Annotations saved to: ./ensemble_results_8label_rougeL_v2/sentence_role_annotations_8label.json
   Total documents annotated: 3000

📊 Role Distribution (8 Labels):
------------------------------------------------------------
  PREAMBLE                 :   7138 ( 1.81%) 
  FACTS                    :  66518 (16.88%) ████████
  ISSUE                    :   5897 ( 1.50%) 
  ARGUMENTS                :   9817 ( 2.49%) █
  REASONING                : 171609 (43.55%) █████████████████████
  PRECEDENT_RELIED         :  16244 ( 4.12%) ██
  PRECEDENT_NOT_RELIED     :      3 ( 0.00%) 
  PROCEDURAL               : 116786 (29.64%) ██████████████
------------------------------------------------------------
  TOTAL                    : 394012
------------------------------------------------------------

STEP 2: CALCULATING ROLE CONTRIBUTION SCORES (8 LABELS)


...

✅ Annotations saved to: ./ensemble_results_8label_rougeL_v2/test_sentence_role_annotations_8label.json
   Total documents annotated: 100

📊 Role Distribution (8 Labels):
------------------------------------------------------------
  PREAMBLE                 :    263 ( 1.96%) 
  FACTS                    :   2185 (16.30%) ████████
  ISSUE                    :    156 ( 1.16%) 
  ARGUMENTS                :    333 ( 2.48%) █
  REASONING                :   6023 (44.92%) ██████████████████████
  PRECEDENT_RELIED         :    541 ( 4.03%) ██
  PRECEDENT_NOT_RELIED     :      0 ( 0.00%) 
  PROCEDURAL               :   3907 (29.14%) ██████████████
------------------------------------------------------------
  TOTAL                    :  13408
------------------------------------------------------------

STEP 5: GENERATING SUMMARIES
        Pipeline: KG-Attention Generation → KG-Diff Refinement
                  → LCS Fusion → Verbatim Span Injection

🔄 Generating summaries...


Processing:   0%|                                                                               | 0/100 [00:00<?, ?it/s]/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_legal_ner_trf' (3.2.0) was trained with spaCy v3.2.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'tran

✅ All summaries generated!

📊 Logs saved to: ./ensemble_results_8label_rougeL_v2/

📊 EVALUATING ALL APPROACHES

  Evaluating BART...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ R1:0.3691  R2:0.1875  RL:0.2102  BS:0.8497

  Evaluating LED...
  ✅ R1:0.4998  R2:0.2649  RL:0.2762  BS:0.8529

  Evaluating PEGASUS...
  ✅ R1:0.3809  R2:0.1889  RL:0.2148  BS:0.8431

  Evaluating LED_kg_attn ✨ (Novel 10)...
  ✅ R1:0.4998  R2:0.2649  RL:0.2762  BS:0.8529

  Evaluating Ensemble Voting...
  ✅ R1:0.4305  R2:0.2165  RL:0.2221  BS:0.8426

  Evaluating Ensemble Weighted...
  ✅ R1:0.4015  R2:0.1997  RL:0.2251  BS:0.8461

  Evaluating Ensemble Best (uses ref)...
  ✅ R1:0.5008  R2:0.2817  RL:0.2845  BS:0.8578

  Evaluating Ensemble Ranking...
  ✅ R1:0.4920  R2:0.2413  RL:0.2377  BS:0.8374

  Evaluating KG-Refined (Novel 5+7)...
  ✅ R1:0.4998  R2:0.2649  RL:0.2762  BS:0.8529

  Evaluating LCS-Fused (Novel 8)...
  ✅ R1:0.5007  R2:0.2646  RL:0.2744  BS:0.8523

  Evaluating Verbatim Injected ✨ (Novel 9)...
  ✅ R1:0.4859  R2:0.2553  RL:0.2663  BS:0.8479

  Evaluating KG-Attn+Full Stack ✨ (Novel 10)...
  ✅ R1:0.4859  R2:0.2553  RL:0.2663  BS:0.8479

📊 FINAL RESULTS — All Approach